# Домашнее задание 8. Мониторинг

**Что делаем:** маленький ML-service + полный контур observability вокруг него.

1. сначала метрики / SLO
2. потом Prometheus + Grafana
3. дальше drift / degradation
4. отдельно DQOps incident
5. в конце Virtual Product Placement со streams

Вся практическая часть лежит в папке `DZ8/`. Контейнеры поднимаются одной командой `docker compose up -d`.


## 1. Определить ключевые бизнес- и технические метрики для ML-системы

Нужно собрать дерево метрик, чтобы было видно не только `service is alive`, но и модель / данные / бизнес.

**Что проверяем:**

- business branch
- application branch
- ML/model branch
- infrastructure branch
- data quality branch
- для каждой ветки: metric / SLI / SLO / owner / action

**Ожидаемый артефакт:** таблица ниже + короткий вывод по выбранному SLO.


**Метрики и SLO**

| branch | metric | SLI | SLO | owner | action |
|---|---|---|---|---|---|
| business | CTR/conversion | product KPI over window | no unexpected drop | product | analyze segment / pause rollout |
| application | p95 latency | PromQL p95 over 5m | `< 1 sec` | backend/MLOps | inspect latency / rollback slow model |
| application | error rate | failed requests ratio | `< 1%` | backend/MLOps | inspect logs / rollback |
| infrastructure | availability | `up{job="ml_service"}` | `> 99%` | DevOps | restart / failover |
| ML/model | drift PSI | Evidently PSI / drift share | `< 0.2` | DS | validate data / retrain decision |
| ML/model | degradation | accuracy/F1 on labelled batch | not worse than baseline by agreed delta | DS | retrain / rollback |
| data quality | DQ incidents | DQOps critical incidents | `0 critical` | data engineer | fix source / backfill / isolate |

**Вывод:**

- основной SLO беру p95 latency < 1 sec для `/predict`
- только latency мало: надо отдельно смотреть app / ML / infra / DQ
- если latency ок, а accuracy падает -> это уже не backend, а модель/данные
- если DQOps ловит schema incident -> сначала чиню источник, а не сразу retrain


## 2. Настроить мониторинг с использованием Prometheus, Grafana, MLflow

В этом ДЗ мониторинг делаю через Prometheus + Grafana.

MLflow server отдельно не поднимаю: в revised ТЗ runtime MLflow убран из scope. Для model-side evidence тут есть `model_version`, `model_drift_psi` + Evidently reports.

**Что проверяем:**

- FastAPI отдает `/metrics`
- Prometheus видит `ml_service` как `UP`
- Grafana dashboard создан через provisioning
- alert `HighLatency` срабатывает после slow-запросов

**Ожидаемый артефакт:** YAML-конфиги + screenshots из `screenshots/`.


In [1]:
%%writefile prometheus.yml
global:
  scrape_interval: 15s
  evaluation_interval: 15s

rule_files:
  - /etc/prometheus/alert_rules.yml

scrape_configs:
  - job_name: ml_service
    metrics_path: /metrics
    static_configs:
      - targets:
          - ml_service:8000


Overwriting prometheus.yml


In [2]:
%%writefile alert_rules.yml
groups:
  - name: dz8_latency_alerts
    rules:
      - alert: HighLatency
        expr: histogram_quantile(0.95, sum(rate(request_latency_seconds_bucket[5m])) by (le)) > 1
        for: 2m
        labels:
          severity: warning
          service: dz8-ml-service
        annotations:
          summary: High request latency detected
          description: p95 latency exceeded 1 second for 2 minutes.


Overwriting alert_rules.yml


**Prometheus / Grafana**

- весь стек стартует из `DZ8/`: `docker compose up -d`
- FastAPI сервис внутри сети compose доступен как `ml_service:8000`
- Prometheus scrape job: `ml_service`
- Grafana datasource/dashboard/alert лежат в `grafana/provisioning/`
- dashboard: `http://localhost:3000/d/dz8-ml/dz-8-ml-service`

**Проверка:**

- `/metrics` содержит `request_latency_seconds_bucket`
- Prometheus target `ml_service` в `UP`
- Grafana строит p95 latency / request rate / target up
- `HighLatency` перешел в `Firing` после slow-запросов

**Скриншоты:**

![Prometheus target UP](screenshots/5.png)

![Grafana dashboard](screenshots/6.png)

![Grafana HighLatency firing](screenshots/8.png)

MLflow server отдельно не поднимаю, т.к. в revised ТЗ он вынесен из runtime scope. Для model monitoring тут есть `model_version`, drift proxy metric + Evidently artifacts.


## 3. Обнаружить деградацию модели и дрифт

Тут беру простой reproducible dataset (`sklearn.datasets.load_wine`).

Схема такая:

1. reference batch -> нормальные данные
2. current batch -> те же признаки, но часть численных колонок специально сдвинута
3. Evidently ловит data drift без labels
4. degradation считаю отдельно: обучаю classifier на reference и проверяю на current

**Ожидаемый артефакт:** `reports/data_drift_report.html`, `reports/data_drift_tests.html`, `reports/degradation_metrics.json`.


In [3]:
import subprocess
import sys

_ = subprocess.run([sys.executable, "scripts/generate_drift_report.py"], check=True)


{
  "baseline_batch": "unchanged holdout batch from sklearn wine",
  "current_batch": "same feature schema with synthetic drift in 3 numeric columns",
  "baseline_acc": 0.9861,
  "current_acc": 0.5972,
  "delta": -0.3889,
  "current_f1_macro": 0.5009,
  "reference_rows": 106,
  "current_rows": 72,
  "drift_note": "data drift is visible without labels; degradation uses labelled current batch"
}


**Drift / degradation**

Запуск:

```bash
python scripts/generate_drift_report.py
```

Что получается:

- `reports/data_drift_report.html`
- `reports/data_drift_tests.html`
- `reports/degradation_metrics.json`

![Evidently drift tests](screenshots/10.png)

![Evidently dataset drift](screenshots/11.png)

Метрики из текущего запуска:

```json
{
  "baseline_acc": 0.9861,
  "current_acc": 0.5972,
  "delta": -0.3889
}
```

**Вывод:**

- reference batch беру как нормальное распределение
- current batch специально сдвинут (synthetic drift), т.е. это demo-стресс
- data drift можно увидеть без labels
- degradation уже считаю по labelled batch: accuracy сильно просела
- concept drift без новых labels нормально не доказать, тут только объясняю разницу


## 4. Обеспечить качество данных с Data Quality Ops

DQOps в этом решении идет через Docker Compose, без локальной установки через Python.

**Что проверяем:**

1. PostgreSQL поднимает таблицу `public.customer_orders`
2. DQOps подключается к этой БД как source
3. checks проходят на нормальной таблице
4. SQL mutation ломает schema contract
5. после повторного run checks должен появиться incident

**Ожидаемый артефакт:** `sql/init_orders.sql`, `sql/break_orders_schema.sql`, screenshot `screenshots/12.png`.


**DQOps incident**

В compose есть PostgreSQL + DQOps:

- PostgreSQL table создается из `sql/init_orders.sql`
- controlled incident лежит в `sql/break_orders_schema.sql`
- DQOps UI: `http://localhost:8888`

Ручной flow:

1. добавить PostgreSQL source (`host=postgres`, `db=dz8`, `user=dqops`, `pass=dqops`)
2. импортировать `public.customer_orders`
3. включить profiling/schema checks
4. run checks на нормальной таблице
5. применить SQL mutation:

```bash
docker exec -i dz8_postgres psql -U dz8 -d dz8 < sql/break_orders_schema.sql
```

6. run checks еще раз -> открыть Incidents / failed checks -> сохранить screenshot

![DQOps schema check](screenshots/12.png)

**Вывод:**

- ломается контракт таблицы: `total_amount` переименован в `total_amount_broken`
- это schema-level incident, т.е. потребитель данных уже не найдет ожидаемую колонку
- response: откатить миграцию / поправить source / сделать backfill, потом rerun checks
- на скрине видно DQOps monitoring check; сам SQL для controlled incident лежит рядом в `sql/break_orders_schema.sql`


## 5. Разработать схему ML-системы для Virtual Product Placement

Задача: описать ML-систему, которая встраивает бренд в видеопоток.

Для этого выбираю Kappa + streams:

- видео режется на кадры / micro-batches
- broker хранит event log
- обработчики можно параллелить
- replay помогает переобработать кадры новой версией модели

**Важно:** это не production video runtime. Реальный YOLO/generative processing заменен demo-обработчиком, чтобы не раздувать ДЗ.

**Ожидаемый артефакт:** `reports/vpp_architecture.png` + stream demo через Redpanda.


**Virtual Product Placement architecture**

![схема VPP](reports/vpp_architecture.png)

Для VPP выбираю Kappa:

- video source -> frame splitter -> event log
- Redpanda/Kafka-compatible broker хранит `frames`
- detector + placement policy выбирают слот/brand
- renderer/inpainting делает вставку
- moderation / brand-safety gate перед output
- processed frames уходят в packager -> CDN/player
- Prometheus/Grafana + drift/DQ checks смотрят latency / lag / failures / входные данные

![пример обработанного кадра](screenshots/9.png)

**Вывод:**

- Lambda тут выглядит лишней, т.к. задача потоковая
- replay из event log помогает переобработать кадры новой версией обработчика
- настоящий YOLO/video runtime не нужен для ДЗ, тут архитектурный proof


**Stream demo**

Kafka-compatible proof сделан через Redpanda:

- topic `frames`
- topic `processed_frames`
- producer: `scripts/vpp_producer.py`
- consumer: `scripts/vpp_consumer.py`
- лог: `reports/vpp_stream_demo.log`

![лог VPP stream demo](screenshots/13.png)

Мини-кусок лога:

```text
# producer
delivered to frames partition 0 offset 30
...

# consumer
processed frame_id=1 -> processed_frames: {"frame_id": 1, "status": "processed", "brand_inserted": "demo_brand", "latency_ms": 41}
```

**Вывод:**

- streams не только на диагармме, они реально запускаются
- обработчик demo-only: он имитирует ML processing, без тяжелого видео
- для ДЗ этого достаточно, потому что проверяем сам stream-подход


**Итого:**

- метрики разделил по веткам: business / app / ML / infra / data quality
- основной SLO: p95 latency < 1 сек (для `/predict`)
- Prometheus видит сервис как `UP`, Grafana строит p95 latency
- alert проверен через slow-запрос (`time.sleep(2)`)
- drift показан через reference/current batch в Evidently
- degradation показана как падение метрики на смещенном batch
- DQOps ловит incident после изменения схемы таблицы (ручной UI-flow, т.к. это часть задания)
- для Virtual Product Placement выбрана Kappa + streams
- чтобы не оставлять streams только на диаграмме, добавлен Redpanda как Kafka-compatible broker
- producer пишет demo-события кадров в `frames`
- consumer имитирует ML-обработку и пишет результат в `processed_frames`

Файлы для проверки: `README.md` / `HW8_Monitoring_НовиковИван.ipynb` / `screenshots/` / `reports/`.
